# 06. Model comparison

All models side by side on the same validation rows and metrics, the final model choice, and the single score on the test set.

## Weighted validation sample: check before use

The LoRA LLM, and possibly TabPFN, are too slow to score all 965,524 validation rows, so they may be scored on all 1,036 laundering rows plus a random 200,000 normal rows, each weighted to stand for 4.8 rows. Before relying on that, the weighted estimate is compared with full validation for the three models already scored in full, over 5 random samples. The rule, fixed before seeing the results: if any weighted PR-AUC differs from the full value by more than 0.01, the slow models are scored on all of validation instead.

In [1]:
# Reload what this section needs from disk, so it runs without the training cells above.
from pathlib import Path

import mlflow
import mlflow.lightgbm
import numpy as np
import pandas as pd

from bankguard.data import load_accounts, load_transactions, split_by_date
from bankguard.evaluate import evaluate, weighted_sample
from bankguard.features import FEATURES, build_features, fit_currency_median
from bankguard.model import predict

DATA = Path("../data")
tx = load_transactions(DATA / "HI-Small_Trans.csv")
accounts = load_accounts(DATA / "HI-Small_accounts.csv")
splits = split_by_date(tx)
train, val = splits["train"], splits["validation"]
X = build_features(tx, accounts, fit_currency_median(train))
X_val, y_val = X.loc[val.index, FEATURES], val["is_laundering"].to_numpy()
del tx, X

mlflow.set_tracking_uri("sqlite:///../mlflow.db")
runs = mlflow.search_runs(
    experiment_names=["bankguard-laundering"],
    filter_string="tags.mlflow.runName = 'LightGBM, owner attributes'",
)
run_id = runs.loc[0, "run_id"]
lgbm = mlflow.lightgbm.load_model(f"runs:/{run_id}/model")
pre = mlflow.artifacts.load_dict(f"runs:/{run_id}/preprocessing.json")

scores = {
    "LightGBM": predict(lgbm, X_val, pre["features"], pre["categories"]),
    "DistilBERT": np.load("../models/distilbert/val_scores.npy"),
    "ModernBERT": np.load("../models/modernbert/modernbert_val_scores.npy"),
}
np.save("../models/lightgbm_val_scores.npy", scores["LightGBM"])
print({name: round(evaluate(y_val, s, name)["pr_auc"], 4) for name, s in scores.items()})

c:\SAP\Projects\Bankguard\bank-guard\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'LightGBM': 0.3412, 'DistilBERT': 0.2281, 'ModernBERT': 0.3023}


In [2]:
full = {name: evaluate(y_val, s, name)["pr_auc"] for name, s in scores.items()}

rows = []
for seed in range(5):
    idx, w = weighted_sample(y_val, n_negatives=200_000, seed=seed)
    for name, s in scores.items():
        estimate = evaluate(y_val[idx], s[idx], name, sample_weight=w)["pr_auc"]
        rows.append({"seed": seed, "model": name, "difference": estimate - full[name]})

check = pd.DataFrame(rows)
print(f"largest difference: {check['difference'].abs().max():.4f} (rule: at most 0.0100)")
check.pivot(index="seed", columns="model", values="difference").round(4)

largest difference: 0.0107 (rule: at most 0.0100)


model,DistilBERT,LightGBM,ModernBERT
seed,,,
0,-0.0009,0.0103,0.0004
1,-0.0075,0.0087,-0.0067
2,0.0037,-0.0014,0.0075
3,-0.0026,0.0050,-0.0041
4,-0.0040,0.0065,0.0107


Over 5 random samples, the weighted PR-AUC differed from the full value by at most 0.0107 (ModernBERT, seed 4; LightGBM reached 0.0103 on seed 0), just over the 0.01 limit fixed in advance. Following the rule, models too slow for a single run are scored on all of validation instead, split across separate runs if needed. The differences are small compared with the gaps between models (0.04 to 0.11 PR-AUC), so the weighted sample would not have changed their ranking, but the rule was set before the check and is kept.